## Graph construction

- Construct NN-graphs w/ $G(V,E)$ with nodes $v_i \in V$ as DAPI coordinates & edges $(e_i, e_j) \in E$ as affinity btw adjacent nodes

- Graph comparisons btw L-slices w/ H&E channels, CyIF channels & joint channels

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd

import networkx as nx
import tifffile

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse

import torch
import torch.nn as nn

from skimage.transform import rescale
from torch_geometric import utils as pyg_utils

import torch
torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
from cellpose import utils as cp_utils
from cellpose import plot as cp_plot
# from cellpose.models import CellposeModel

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')
import IO, utils, plot, gen_graph

### Segmentation & Graph Construction

#### CyIF channels

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellposeModel(
    gpu=device,
    model_type='nuclei',
    # pretrained_model=pretrained_model.value
)

In [ ]:
# Load dataset with clipped ROI (aligned CyCIFs)
data_path = '../data/cycif/zonation_hires/'
filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif' and 'series' not in f]
dapis = [tifffile.imread(os.path.join(data_path, f))[0] 
         for f in filenames]
gc.collect()


In [ ]:
out_path = '../data/cycif/segment/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for f, img in zip(filenames, dapis):
    print('Segmentation on image {}...'.format(f))
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min())

    mask = model.eval(
        img,
        diameter=8,
        tile=True,
        channels=[0, 0],
        flow_threshold=1,
        min_size=50
    )[0]
    torch.cuda.empty_cache()

    tifffile.imwrite(os.path.join(out_path, f.split('.')[0] + '_mask.tif'), mask)
    

Construct k-NN graphs:
- Try not subsampling graph upon initialization
- Graph aggregation w/ convolution operators

In [ ]:
img_path = '../data/cycif/zonation_hires/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']
dapis = [tifffile.imread(os.path.join(img_path, f))[0]
        for f in filenames]
masks = [tifffile.imread(os.path.join(mask_path, f.split('.')[0]+'_mask.tif'))
         for f in filenames]


In [ ]:
# Save simplified "meta-graphs" per L-slice
out_path = '../data/cycif/graph/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

# prop_nodes_to_keep = 0.1

for f, mask in zip(filenames, masks):
    print('Computing graph for {}...'.format(f))
    coords = gen_graph.get_centroids(mask)    
    G = gen_graph.construct_graph(coords, k=5, r=10)
    pickle.dump(G, open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'wb'))

    del G
    gc.collect()

In [ ]:
Gs = []
for f, dapi in zip(filenames, dapis):
    G =  pickle.load(open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'rb'))
    Gs.append(G)
    del G, dapi

L = len(Gs)
n_nodes = [len(G) for G in Gs]

fig, ax = plt.subplots(figsize=(8, 2))
ax.plot(np.arange(L), n_nodes, '.-')
ax.set_xlabel('# section')
ax.set_ylabel(r"$\vert V \vert$")

plt.show()

del L

In [ ]:
plot.disp_network(dapis[0], Gs[0], figsize=(15, 15), node_size=0.2, edge_width=0.2)

In [ ]:
plot.disp_graph_overlaps(Gs[3:6], labels=filenames[3:6], figsize=(8, 8), 
                         node_size=0.5, edge_width=0.1)

Feature matrix extraction

In [ ]:
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

img_path = '../data/cycif/zonation_hires/'
graph_path = '../data/cycif/graph/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']

# Load images
imgs_rgb = [tifffile.imread(os.path.join(img_path, f))
            for f in filenames]

# Load graphs
Gs = [pickle.load(open(os.path.join(graph_path, f), 'rb'))
      for f in sorted(os.listdir(graph_path))
      if f[-3:] == 'pkl']

# Retrieve graph coords
coords_list = [gen_graph.get_coords_from_graph(G) for G in Gs]

# Convert channel expressions to [0, 1] & empty channels to 0s
imgs = []
for img in imgs_rgb:
    img_adj = img.copy().astype(np.float32)
    for i, chan in enumerate(img):
        if chan.max() <= 1:
            img_adj[i] = 0
        else:
            img_adj[i] = (img_adj[i] - img_adj[i].min()) / (img_adj[i].max() - img_adj[i].min())
    imgs.append(img_adj)
del imgs_rgb

features = [utils.znorm(gen_graph.construct_feature_matrix(img, coords, r=5))
            for img, coords in zip(imgs, coords_list)]

In [ ]:
plot.disp_corr_features(features, labels=cyif_chans, titles=filenames, ncols=6)

In [ ]:
# Save feature 
graph_path = '../data/cycif/graph/'
for f, feature in zip(filenames, features):
    feature_df = pd.DataFrame(feature, columns=cyif_chans)
    feature_df.to_csv(os.path.join(graph_path, f.split('.')[0] + '.csv'), index=True)

Benchmark: PCA baseline

In [ ]:
# Baseline: PCA??
from sklearn.decomposition import PCA

# TODO:
# Load sample_graph & sample_features
sample_features = features[0]
sample_graph = Gs[0]

pca = PCA(n_components=2)
pc = pca.fit_transform(sample_features)

plot.disp_network_embedding(img[3], sample_graph, embedding=pc[:, 0], alpha=0.25,
                       node_size=1, edge_width=0.2, figsize=(8, 8),
                       title='ASS vs. PC2')
                       

In [ ]:
plt.figure(figsize=(3, 1.5))
plt.hist(pc[:, 0], bins=60, edgecolor='white')
plt.title('PC1')
plt.show()

from scipy.stats import pearsonr
plt.scatter(z.squeeze(), pc[:, 0], s=1)
plt.xlabel('Learnt U')
plt.ylabel('PC1')
plt.title('R={}'.format(
    np.round(pearsonr(z.squeeze(), pc[:, 0])[0], 4)
))

#### Exploratory

Construct CyCIF k-NN graphs across nucleid centroids (nodes)

In [ ]:
from cellpose.models import CellposeModel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellposeModel(
    gpu=device,
    model_type='nuclei',
    # pretrained_model=pretrained_model.value
)

In [ ]:
dapi = tifffile.imread('../data/desi/trimmed_cycif.tif')[0]
dapi = (dapi - dapi.min()) / (dapi.max() - dapi.min())
dapi.shape

mask = model.eval(
    dapi,
    diameter=5,
    tile=True,
    channels=[0, 0],
    flow_threshold=4,
    min_size=30
)[0]

In [ ]:
coords = gen_graph.get_centroids(mask)
G = gen_graph.construct_graph(coords, k=10, r=50)

#G = sample_nodes(G_raw, k=30, sample_ratio=0.2)
#del G_raw, coords
# coords = get_coords_from_graph(G)  # Get coords from sampled graph

pickle.dump(G, open('../data/desi/cyif_graph.pkl', 'wb'))
gc.collect()


In [ ]:
coords = gen_graph.get_coords_from_graph(G)
img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[0,1,2,6]].astype(np.float32)

img_normed = img.copy()
for i, chan in enumerate(img):
    img_normed[i] = (chan - chan.min()) / (chan.max() - chan.min())
feature_mat = utils.znorm(gen_graph.construct_feature_matrix(img_normed, coords, r=3))

feature_mat_df = pd.DataFrame(feature_mat, columns=['DAPI', 'ASS1', 'GS', 'CYP'])
feature_mat_df.to_csv('../data/desi/cyif_feature.csv', index=True)

Temperature inference on CyIF:

In [ ]:
img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[0,1,2,6]].astype(np.float32)
G = pickle.load(open('../data/desi/cyif_graph.pkl', 'rb'))
coords = gen_graph.get_coords_from_graph(G)
feature_mat = pd.read_csv('../data/desi/cyif_feature.csv', index_col=[0]).values

graph_data = pyg_utils.from_networkx(G)
edge_index, edge_weight = graph_data.edge_index, graph_data.weight

edge_index, edge_weight = utils.nx_to_edge_attrs(G)
x = torch.tensor(feature_mat, dtype=torch.float32)
# adj_mat = pyg_utils.to_torch_csr_tensor(edge_index=edge_index, edge_attr=edge_weight)

In [ ]:
plot.disp_network(img[0], G, figsize=(12, 12), title='Original G')

In [ ]:
topk_pool = gen_graph.TopKPooling(4, ratio=0.5, nonlinearity='sigmoid')

x, ei, ew, _, perm, _ = topk_pool(x=x, edge_index=edge_index, edge_attr=edge_weight)
perm = perm.detach().numpy()
adj_matrix_pooled, G_pooled = gen_graph.pooled_edge_attrs_to_graph(ei,
                                                                   ew,
                                                                   G=G,
                                                                   perm=perm,
                                                                   to_nx=True)

plot.disp_network(img[0], G_pooled, figsize=(12, 12), title='Pooled G (topK)')


---

### VGAE w/ Hierarchical representations

- Dim-reduction on features (`z`: representation, `u`: interpretability)<br> $z \in \mathbb{R^{N' \times 10}}$, $u \in \mathbb{R}^{N' \times 1}$
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- TODO: Benchmark against K-means clustering & PCA

In [ ]:
import vgae, utils
import model_train, model_eval, configs, dataset, zonation

from torch_geometric.loader import DataLoader
from torch_geometric.data import ClusterData

In [ ]:
# Load DESI normed image

# desi_raw = tifffile.imread('../data/desi/Desi_neg_7.tif').astype(np.float32)
# desi_img = desi_raw.transpose(0,2,1)[:, 82:222, 60:200]  # Trim & rotate/flip & resize to CyIF patch
# desi_normed = np.zeros_like(desi_img)
# for i in range(desi_img.shape[0]):
#     desi_normed[i] = (desi_img[i] - desi_img[i].min()) / (desi_img[i].max() - desi_img[i].min())

# tifffile.imwrite('../data/desi/Desi_neg_7_aligned.tif', desi_normed)

desi_img = tifffile.imread('../data/desi/aligned/Desi_neg_7_aligned.tif')

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_prior = tifffile.imread('../data/desi/cyif_prior/Desi_neg_7_aligned.tif')  # [-1, 1]
u_prior = torch.tensor(u_prior)
yy, xx = np.meshgrid(np.arange(desi_img.shape[-2]), np.arange(desi_img.shape[-1]))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T

# Convert desi_image to [C, X, Y] to
# extract features by row (as the inner ptr for "flatten")
nchans = desi_img.shape[0]
desi_feature_mat = desi_img.transpose(0,2,1).reshape(nchans, -1).T  
desi_feature_mat = utils.norm_features(desi_feature_mat)


# Testing robustness w/ different graph connections
G_desi = gen_graph.construct_graph(desi_coords, k=10, r=5)  

# plot.disp_network(desi_img[7], G_desi, figsize=(12, 12), node_size=1, edge_width=0.5)

Create sub-graphs with mini-batches

In [ ]:
ims_loader = dataset.DESIGraphDataset(data_path='../data/desi/aligned/',
                                       prior_path='../data/desi/cyif_prior/',
                                       n_subgraphs=8)
ims_data = ims_loader.load_graphs()
ims_data = DataLoader(ims_data)

subgraphs = []
for x in ims_data:
    subgraph = pyg_utils.to_networkx(x, node_attrs=['pos'], to_undirected=True)
    subgraphs.append(subgraph )

del x, subgraph

In [ ]:
# DEBUG model
from importlib import reload
reload(vgae)
reload(utils)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)
# reload(zonation)

In [ ]:
lr = 1e-3
n_epochs = 50

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          lr=lr)

model_configs = configs.set_model_configs(beta_prior=0.1,
                                          c_in=nchans,
                                          c_hidden=8,
                                          c_latent=1,
                                          pz_scale=1,
                                          pu_scale=1)
model = vgae.SparseVGAE(model_configs)

# model_configs = configs.set_model_configs(beta_prior=0.5,
#                                           c_in=nchans,
#                                           c_hidden=16,
#                                           c_latent=1,
#                                           pz_scale=5,
#                                           pu_scale=1)
# model = vgae.SparseGamVGAE(model_configs)

device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, sparse_losses, kls, signs = model_train.train(model, ims_data, train_configs)

In [ ]:
del model, device, optimizer

In [ ]:
# Check learnt u_std & z_std, prior set too large so that posterior `u` deviates too much from the prior constraint
print(model.decoder.pu_scale)
print(model.decoder.pz_scale)

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 18))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), sparse_losses)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Smooth regularization')

ax4.plot(np.arange(n_epochs), kls)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('KLs')

ax5.plot(np.arange(n_epochs), signs)
ax5.set_xlabel('Epochs')
ax5.set_ylabel('Sign')

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model, '../data/desi/model.pt')

In [ ]:
# Graph reconstruction checks 

# subgraph
# A = nx.adjacency_matrix(subgraph).toarray()
# latent = model_eval.eval(model, subgraph, graph_data.x)
# reconst = model.decoder(latent)

# full graph
A = nx.adjacency_matrix(G_desi).toarray()

latent = model_eval.eval(model, G_desi, desi_feature_mat)
reconst = model.decoder(latent)

z = latent.qz.detach().cpu().numpy()
u_posterior = latent.qu_loc.detach().cpu().numpy()
A_hat = reconst.A_hat.detach().cpu().numpy()

In [ ]:
# plot distributions of u, z & A_hat
plt.figure(figsize=(4, 2))
plt.hist(z.flatten().squeeze(), edgecolor='white', bins=10)
plt.show()

In [ ]:
plt.imshow(A[:256, :256], cmap='magma')

In [ ]:
plt.imshow(A_hat[:256, :256], cmap='magma')

In [ ]:
# Visuzlize Z's
for i, zj in enumerate(z.T):
    plot.disp_network_embedding(desi_img.mean(0), G_desi, zj.squeeze(), alpha=0.5, 
                        node_size=10, edge_width=0.5, fontsize=64,
                        figsize=(15, 15), title='Z-factor {}'.format(i))

In [ ]:
sns.clustermap(z[:1000, ])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
sns.heatmap(np.corrcoef(z.T), cmap='coolwarm', ax=ax, square=True)
fig.suptitle('Correlations btw z-factors')
plt.show()


In [ ]:
# Note: coords saved in [x,y]-index in graph, but in [i,j]-index in image!!:
# transpose 2D image before flatten
plot.disp_network_embedding(desi_img.mean(0), G_desi, embedding=u_prior.T.flatten(), alpha=0.5, 
                       node_size=10, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U prior (CyCIF)')

In [ ]:
plot.disp_network_embedding(desi_img.mean(0), G_desi, embedding=u_posterior.flatten(), alpha=0.5, 
                       node_size=10, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U posterior')

In [ ]:
# Save U posterior
np.save('../data/desi/u_posterior.npy', u_posterior)

In [ ]:
# Create discretized zonation maps from posterior `u`
u_posterior_map = u_posterior.reshape(-1, desi_img.shape[-1]).T
u_posterior_discretized_map = utils.infer_zones(u_posterior_map, n_layers=10)

plt.figure()
plt.imshow(u_posterior_discretized_map, cmap='turbo')
plt.colorbar()
plt.title('Posterior Temperature')
plt.axis('off')
plt.show()

Visualize DESI gradients sorted on posterior `U`:

In [ ]:
# Load DESI m/z labels:
ifile = open('../data/desi/Desi_neg_7.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/desi/Desi_neg_7.tif')
ifile.close()

In [ ]:
# Visualize DESI features sorted by prior temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_prior.T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                        ion_labels, 
                        title='Prior Temperature', nbins=100)


In [ ]:
# Visualize DESI features sorted by POSTERIOR temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_posterior.squeeze())] # sort by posterior temp

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels,
                         title='Posterior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=100)

**TODO: remove duplicated DESI channel - 415.31352 m/z ± 50 mDa**

In [ ]:
nbins = 100
# Feature x nbins
desi_binned_grads, desi_binned_stds = utils.get_binned_features(desi_feature_mat_sorted, nbins=nbins) 
desi_binned_grads, desi_binned_stds = desi_binned_grads.T, desi_binned_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_binned_grads.argmax(1)
sorted_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
desi_binned_grads = desi_binned_grads[np.argsort(argmax_grad), :]
desi_binned_stds = desi_binned_stds[np.argsort(argmax_grad), :]

desi_binned_grads_df = pd.DataFrame(desi_binned_grads,  
                                    index=sorted_ion_labels,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
# desi_binned_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients.csv', index=True)
del nbins

In [ ]:
plot.disp_desi_gradients(desi_binned_grads_df.values.T, 
                         sorted_ion_labels, cluster_ions=False,
                         title='Posterior Temperature', nbins=100)

In [ ]:
from scipy.stats import linregress

nbins = 100
desi_gradient_ts = []  # Fitted slope of the features along PV -> CV trajectory

for feature_mean, feature_std, label in zip(desi_binned_grads, desi_binned_stds, sorted_ion_labels):
    plot.disp_desi_gradient(feature_mean, feature_std, title=label)
    ts = linregress(np.linspace(-1, 1, nbins), feature_mean)
    desi_gradient_ts.append(ts)

del label, feature_mean, feature_std

In [ ]:
# Summarize expression gradienst along PV->CV trajectory
desi_gradient_slopes = [ts.slope for ts in desi_gradient_ts]
desi_gradient_pvals = [ts.pvalue for ts in desi_gradient_ts]
desi_gradient_categories = []
for slope, pval in zip(desi_gradient_slopes, desi_gradient_pvals):
    if pval >= 0.01:
        desi_gradient_categories.append('Uncertain')
    elif slope > 0:
        desi_gradient_categories.append('+')
    else:
        desi_gradient_categories.append('-')


In [ ]:
valid_slopes = [slope 
                for (slope, p) in zip(desi_gradient_slopes, desi_gradient_pvals)
                if p < 0.01]

plt.figure(figsize=(5, 2))
plt.hist(valid_slopes, edgecolor='white', bins=10)
plt.xlabel(r'Fitted slope $\beta$')
plt.suptitle('Metabolite gradients along PV->CV trajectory')
plt.show()

del valid_slopes

In [ ]:
# summarize metabolites' gradient
gradient_df = pd.DataFrame({
    'Label': ion_labels,
    'Slope': desi_gradient_slopes,
    'Category': desi_gradient_categories,
    'p-val': desi_gradient_pvals
})

gradient_df.set_index('Label', inplace=True)
gradient_df.head()

In [ ]:
gradient_df.to_csv('../data/desi/metabolite_gradients.csv', index=True)

Modality integration **Using CyIF as priors to regularize u**:

- Vanilla design: (1). distance transformation to closest, **binarized** GS signal; (2). Graph-based heat diffusion

In [ ]:
from skimage.filters import threshold_otsu
from skimage.filters import gaussian as gaussian_blur
from skimage.morphology import binary_erosion
from skimage.transform import resize
from scipy import ndimage as ndi

import zonation


In [ ]:
def get_gs_dist(gs, erosion_size=7):
    assert 0 <= gs.min() <= gs.max() <= 1, \
        "Single-channel intensity should be normalized to [0, 1]"

    thresh = threshold_otsu(gs)
    gs_bin = (gs > thresh).astype(np.uint8)
    gs_bin = binary_erosion(gs_bin, np.ones((erosion_size, erosion_size)))
    gs_dist = ndi.distance_transform_edt(1-gs_bin)
    return gs_dist


def create_vein_mask(gs, ass, q=0.25, sigma=1.5):    
    # Binarize GS & ASS to obtain CV / PV approximation 
    gs_blur = gaussian_blur(gs, sigma=sigma)
    thresh = np.quantile(gs_blur, 1-q)
    cv_prior = (gs > thresh).astype(np.uint8)

    ass_blur = gaussian_blur(ass, sigma=sigma)
    thresh = np.quantile(ass_blur, q)
    pv_prior = (ass < thresh).astype(np.uint8)

    u_prior = np.zeros_like(gs, dtype=np.int8)
    u_prior[np.logical_and(cv_prior == 0, pv_prior == 1)] = -1
    u_prior[np.logical_and(cv_prior == 1, pv_prior == 0)] = 1

    return u_prior

In [ ]:
cyif_img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[2, 6, 1]].astype(np.float32)  # Only load GS, CYP & ASS

for i in range(len(cyif_img)):
    cyif_img[i] = (cyif_img[i]-cyif_img[i].min()) / (cyif_img[i].max()-cyif_img[i].min())
# Intenstity: [0, 1]

In [ ]:
# create `u_prior` via graph-based heat diffusion
cyif_img_ds = resize(cyif_img, 
                     output_shape=(cyif_img.shape[0], desi_img.shape[-2], desi_img.shape[-1]),
                     preserve_range=True)

vein_mask = create_vein_mask(gs=cyif_img_ds[0], ass=cyif_img_ds[-1], q=0.05)
diff_model = zonation.HeatDiffusion(vein_prior=vein_mask, ndim=2)
_, _ = diff_model.get_interior_U()
u_prior = diff_model.infer_zone_dynamics()

# plt.imshow(u_prior, cmap='coolwarm')
# plt.colorbar()
# plt.show()

tifffile.imwrite('../data/desi/cyif_prior_heat.tif', u_prior)


---